In [1]:
!pip install -q openai

In [2]:
import json, os, re, time, random
from google.colab import userdata
from openai import OpenAI

API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o"

In [3]:
import json, os, re, time, random
from google.colab import userdata
from openai import OpenAI

API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o"

In [4]:
with open('RPT_SEED_DATA.json', encoding='utf-8') as f:
    seed_data = json.load(f)

print(f"시드 개수: {len(seed_data)}")
existing_ids = set(item['id'] for item in seed_data)

시드 개수: 30


In [5]:
RPT_DESCRIPTION_MAP = {
    "RPT-01-01": {
        "id": "RPT-01-01",
        "chapter": "제1장 사업개요",
        "section": "제1절 개요",
        "description": "사업의 기본 식별정보와 범위를 개괄하는 절이다. 사업명·주관기관·사업자·예산·범위가 계획과 일치하는지 평가한다.",
        "standard_structure": ["사업명", "주관기관 및 이용기관", "사업자", "사업예산", "사업의 범위"],
        "quality": ["완전성", "정확성"],
        "PEP_mapping": ["PEP-01", "PEP-02", "PEP-04-01"],
        "PEP_mapping_note": "사업명, 사업범위, 사업기간"
    },
    "RPT-01-02": {
        "id": "RPT-01-02",
        "chapter": "제1장 사업개요",
        "section": "제2절 사업의 배경 및 목적",
        "description": "사업 추진의 배경·목적과 성과계획을 기술하는 절이다. 목적 구조의 완전성·정확성과 성과지표의 검증가능성을 평가한다.",
        "standard_structure": ["사업 추진 배경", "사업 목적", "성과 계획"],
        "quality": ["완전성", "정확성", "검증가능성"],
        "PEP_mapping": ["PEP-03-02", "PEP-05-01"],
        "PEP_mapping_note": "사업목적, 사업추진체계"
    },
    "RPT-01-03": {
        "id": "RPT-01-03",
        "chapter": "제1장 사업개요",
        "section": "제3절 사업추진체계",
        "description": "사업 수행을 위한 추진체계와 조직 구성을 기술하는 절이다. 계획된 추진조직이 빠짐없이 반영됐는지 평가한다.",
        "standard_structure": ["총괄조직도", "조직별 역할", "사업자 추진조직"],
        "quality": ["완전성"],
        "PEP_mapping": ["PEP-05-02"],
        "PEP_mapping_note": "사업추진체계"
    },
    "RPT-01-04": {
        "id": "RPT-01-04",
        "chapter": "제1장 사업개요",
        "section": "제4절 추진경과",
        "description": "사업의 단계별 추진경과를 기술하는 절이다. 계획된 절차·투입인력이 이행됐는지와 정량 근거를 평가한다.",
        "standard_structure": [],
        "quality": ["완전성", "검증가능성"],
        "PEP_mapping": ["PEP-09", "PEP-06"],
        "PEP_mapping_note": "투입인력계획, 사업추진절차"
    },
    "RPT-02-01": {
        "id": "RPT-02-01",
        "chapter": "제2장 사업내용(개발사업)",
        "section": "제1절 적용방법론",
        "description": "사업에 적용한 개발방법론과 추진절차·산출물을 기술하는 절이다. 계획 단계·산출물과의 대응 및 정량 근거를 평가한다.",
        "standard_structure": ["방법론 적용내용", "사업추진절차", "산출물"],
        "quality": ["완전성", "추적성", "검증가능성"],
        "PEP_mapping": ["PEP-07", "PEP-06", "RPT-01-04"],
        "PEP_mapping_note": "산출물 계획, 사업추진절차, 추진경과"
    },
    "RPT-02-02": {
        "id": "RPT-02-02",
        "chapter": "제2장 사업내용(개발사업)",
        "section": "제2절 개발내용",
        "description": "개발 범위와 세부 개발내역, 통합·연계 결과, 원시자료 내용을 기술하는 절이다. 계획 과업의 이행 여부, 수치 정합성, 검증가능성, 추진경과와의 추적성을 평가한다.",
        "standard_structure": ["개발범위", "세부 개발내역", "통합/연계 결과", "원시자료 내용 및 제공기관"],
        "quality": ["완전성", "검증가능성", "추적성", "정확성"],
        "PEP_mapping": ["PEP-04-01", "PEP-12", "PEP-14", "RPT-01-04"],
        "PEP_mapping_note": "사업범위(개발 대상 업무), 품질보증, 보안대책, 추진경과"
    },
    "RPT-02-03-01": {
        "id": "RPT-02-03-01",
        "chapter": "제2장 사업내용(개발사업)",
        "section": "제3절 시스템 구성도",
        "description": "전체 시스템 구성도와 H/W, S/W, 통신망, 데이터베이스 구성내역을 기술하는 절이다. 계획된 개발·운영환경이 누락 없이 반영됐는지와 구성 정보가 계획과 모순 없는지 평가한다.",
        "standard_structure": ["전체 시스템 구성도", "H/W 및 S/W 구성내역", "통신망 구성도", "데이터베이스 구성도"],
        "quality": ["완전성", "정확성"],
        "PEP_mapping": ["PEP-04-02"],
        "PEP_mapping_note": "사업 범위(개발/운영환경)"
    },
    "RPT-02-04-01": {
        "id": "RPT-02-04-01",
        "chapter": "제2장 사업내용(개발사업)",
        "section": "제4절 표준화 적용결과",
        "description": "정보화업무표준, 정보화기반표준, 상호운용성 확보 등을 위한 기술평가 검토, 행정정보데이터베이스 표준화 적용 결과를 기술하는 절이다. 계획된 표준화 항목이 결과보고서에 누락 없이 대응되는지, 규격·코드체계·명명규칙 등이 계획과 모순 없는지, 적용 결과가 측정 가능한 근거로 제시됐는지 평가한다.",
        "standard_structure": [
            "정보화업무표준",
            "정보화기반표준",
            "상호운용성 확보 등을 위한 기술평가 검토",
            "행정정보데이터베이스 표준화"
        ],
        "quality": ["정확성", "검증가능성", "완전성"],
        "PEP_mapping": ["PEP-11-01", "PEP-11-02", "PEP-11-03", "PEP-11-04"],
        "PEP_mapping_note": "표준화 계획"
    },
    "RPT-02-05": {
        "id": "RPT-02-05",
        "chapter": "제2장 사업내용(개발사업)",
        "section": "제5절 보안 부문",
        "description": "시스템 보안 요건과 개인정보보호 현황 및 대책을 기술하는 절이다. 계획된 보안대책이 누락 없이 반영됐는지, 보안 적용 내용이 계획과 모순 없는지, 점검 결과가 측정 가능한 근거로 제시됐는지 평가한다.",
        "standard_structure": ["시스템 보안 요건", "개인정보보호 현황 및 대책"],
        "quality": ["정확성", "완전성", "검증가능성"],
        "PEP_mapping": ["PEP-14"],
        "PEP_mapping_note": "보안대책"
    },
    "RPT-02-06": {
        "id": "RPT-02-06",
        "chapter": "제2장 사업내용(개발사업)",
        "section": "제6절 법제도 정비실적",
        "description": "시스템 구축 과정에서 부각시킬 수 있는 법제도 정비 내용과 예산, 일정, 타 기관 협조 등의 한계로 부족했던 부문을 기술하는 절이다. 결과보고서 자체에 법제도 정비 실적과 한계가 측정 가능하거나 확인 가능한 근거로 제시됐는지 평가한다.",
        "standard_structure": [
            "시스템 구축에 있어 부각시킬 수 있는 내용",
            "시스템 구축시 예산, 일정, 타 기관간 협조 등의 한계로 부족한 부문"
        ],
        "quality": ["검증가능성"],
        "PEP_mapping": [],
        "PEP_mapping_note": ""
    },
    "RPT-03-01": {
        "id": "RPT-03-01",
        "chapter": "제3장 운영계획 및 발전방향",
        "section": "제1절 운영계획",
        "description": "운영조직 및 체계, 시스템 보안대책, 예산확보계획을 기술하는 절이다. 운영계획이 조직, 예산, 보안대책 등 확인 가능한 근거와 함께 제시됐는지 평가한다.",
        "standard_structure": ["운영조직 및 체계", "시스템 보안대책", "예산확보계획"],
        "quality": ["검증가능성"],
        "PEP_mapping": [],
        "PEP_mapping_note": ""
    },
    "RPT-03-02": {
        "id": "RPT-03-02",
        "chapter": "제3장 운영계획 및 발전방향",
        "section": "제2절 발전방향",
        "description": "시스템 활성화를 위한 타 시스템 연계 또는 중장기 확대방안을 기술하는 절이다. 발전방향이 일정, 대상 시스템, 단계, 범위 등 확인 가능한 근거로 제시됐는지 평가한다.",
        "standard_structure": ["시스템 활성화를 위한 타 시스템 연계 또는 중장기 확대방안"],
        "quality": ["검증가능성"],
        "PEP_mapping": [],
        "PEP_mapping_note": ""
    }
}

print(f"RPT_DESCRIPTION_MAP 항목 수: {len(RPT_DESCRIPTION_MAP)}")

RPT_DESCRIPTION_MAP 항목 수: 12


In [6]:
DEFECT_PREFIX = {"완전성": "완전", "정확성": "정확", "검증가능성": "검증", "추적성": "추적"}
TRACE_TYPES = ["① 오소·고아 산출물", "② 단계 재구성"]

def build_combos():
    combos = []
    for code_, desc in RPT_DESCRIPTION_MAP.items():
        quality = desc.get('quality') or []
        if not quality:
            continue
        combos.append({"desc_code": code_, "target_criteria": list(quality), "target_defect_type": "충족", "trace_type": None})
        for c in quality:
            if c == "추적성":
                for tt in TRACE_TYPES:
                    combos.append({"desc_code": code_, "target_criteria": list(quality), "target_defect_type": "추적불가", "trace_type": tt})
                combos.append({"desc_code": code_, "target_criteria": list(quality), "target_defect_type": "추적검토", "trace_type": None})
            else:
                prefix = DEFECT_PREFIX[c]
                combos.append({"desc_code": code_, "target_criteria": list(quality), "target_defect_type": f"{prefix}불가", "trace_type": None})
                combos.append({"desc_code": code_, "target_criteria": list(quality), "target_defect_type": f"{prefix}검토", "trace_type": None})
    return combos

combos = build_combos()
random.seed(42)
random.shuffle(combos)
print(f"조합 개수: {len(combos)}")

조합 개수: 66


In [7]:
TARGET_TOTAL = 150
BATCH_N = 5
MAX_CALLS = 60

In [13]:
SYSTEM_PROMPT = """당신은 결과보고서(RPT) 평가 학습데이터를 증강하는 생성기입니다.
목표는 기존 시드데이터와 동일한 스키마·품질 기준으로 새로운 {"id", "input", "output"} 예시를 생성하는 것입니다.

이 프롬프트에서는 절별 평가기준 매트릭스를 직접 사용하지 않습니다.
평가기준 조합과 PEP 매핑은 코드에서 관리되는 target_description 객체가 이미 결정합니다.

[입력]
- target_description: 코드에서 관리되는 RPT_DESCRIPTION_MAP 중 하나의 description 객체
  - id: 평가 대상 RPT 절 코드
  - chapter: 결과보고서 장 명칭
  - section: 결과보고서 절 명칭
  - description: 해당 절의 평가 목적 설명
  - standard_structure: 해당 절에서 확인해야 할 하위 구성요소 목록
  - quality: 해당 절에 허용된 평가기준 목록
  - PEP_mapping: 대조할 PEP 코드 목록
  - PEP_mapping_note: 사람이 이해하기 위한 PEP 매핑 참고 설명

- target_criteria: 이번 예시에서 실제로 다룰 평가기준
  - 반드시 target_description.quality 안에 포함된 기준만 사용합니다.
  - 예: ["완전성"], ["정확성"], ["검증가능성"], ["완전성", "정확성"]

- target_defect_type: 목표 결함 유형
  - 충족
  - 완전불가
  - 정확불가
  - 검증불가
  - 추적불가
  - 완전검토
  - 정확검토
  - 검증검토
  - 추적검토

- trace_type: 추적성을 다룰 경우의 불가 유형
  - "① 오소·고아 산출물"
  - "② 단계 재구성"
  - 추적성을 다루지 않으면 생략합니다.

- domain_hint: 선택 입력
  - 예: 교육청, 지자체, 공공기관, 보건, 복지, 도서관, 교통, 시설관리, 민원, 표준화, 보안 등

[description 사용 규칙]
- 생성 요청 시 전체 매트릭스가 아니라 target_description 객체가 입력됩니다.
- target_description은 코드에서 관리되는 RPT_DESCRIPTION_MAP에서 선택된 값입니다.
- 생성기는 target_description에 포함된 RPT 코드, 장, 절, 하위 구성요소, 평가기준, PEP 매핑만 사용합니다.
- target_description.quality에 없는 평가기준은 임의로 추가하지 않습니다.
- target_description.PEP_mapping에 없는 PEP 항목은 임의로 추가하지 않습니다.
- target_description의 description, standard_structure, PEP_mapping_note는 생성 범위 설정용 메타데이터이며, 최종 코멘트의 원문 근거로 인용하지 않습니다.
- target_description.PEP_mapping이 비어 있는 절은 PEP 대조가 아니라 RPT 자체의 검증가능성 중심으로만 작성합니다.
- 법제도 정비실적, 운영계획, 발전방향처럼 PEP_mapping이 없는 절은 완전성·정확성·추적성을 생성하지 않고, target_description.quality에 있는 검증가능성만 생성합니다.

[평가기준명 정규화]
평가기준명은 반드시 아래 네 가지 중 하나만 사용합니다.
- 완전성
- 정확성
- 검증가능성
- 추적성

정규화 규칙:
- "완전"은 "완전성"으로 변환합니다.
- "정확"은 "정확성"으로 변환합니다.
- "검증"은 "검증가능성"으로 변환합니다.
- "연결"은 "추적성"으로 변환합니다.
- "코드" 또는 "-"는 LLM 평가기준이 아니므로 생성 대상에서 제외합니다.

[생성 절차 — 반드시 순서대로]

1단계. 가상의 공공 정보화사업 설정
- 지자체/교육청/공공기관 + 사업 유형을 새로 조합합니다.
- 기존 시드와 대상기관 수, 사업 유형, 소재 도메인이 겹치지 않게 합니다.
- 사업 도메인 힌트가 주어진 경우 그 범위 안에서 새 사업을 만듭니다.

2단계. PEP_excerpt 작성
- PEP_excerpt는 사업수행계획서 원문처럼 작성합니다.
- RFP 원문처럼 쓰지 않습니다.
- target_description.PEP_mapping에 포함된 PEP 항목만 작성합니다.
- target_description.PEP_mapping에 없는 PEP 항목을 임의로 추가하지 않습니다.
- target_description.standard_structure와 연결되는 계획 항목을 구체적 수치·목록·기준과 함께 작성합니다.
- 완전성을 다룰 경우, 단일값 필드(사업명/기간/예산/기관명)와 목록형 과업·업무범위 항목을 명확히 구분해서 작성합니다.
- 완전성 판단의 재료가 되는 목록형 항목은 최소 4~5개 이상 작성합니다.
- 정확성을 다룰 경우, 나중에 RPT_excerpt에서 왜곡할 수 있는 구체 수치·규격·버전·등급·기간을 반드시 포함합니다.
- 검증가능성을 다룰 경우, 계획 단계의 정량 목표·점검 기준·측정 방법 중 하나 이상을 포함합니다.
- 추적성을 다룰 경우, 단계명·과업명·Task·일정·산출물명이 서로 연결될 수 있도록 작성합니다.
- 필드명은 반드시 PEP_excerpt로 고정합니다.

단, target_description.PEP_mapping이 비어 있는 절의 경우:
- PEP_excerpt는 "해당 절은 PEP 대조 대상이 아니며 RPT 자체 검증가능성 중심으로 평가한다"는 짧은 설명만 작성하거나 생략 가능한 수준으로 작성합니다.
- 이 경우 완전성·정확성·추적성 시나리오를 만들지 않습니다.

3단계. RPT_excerpt 작성 — target_defect_type에 맞게 설계
- RPT_excerpt는 사업추진결과보고서 원문처럼 작성합니다.
- target_description.chapter와 target_description.section에 해당하는 절의 결과보고서 내용으로 작성합니다.
- target_description.standard_structure에 포함된 하위 구성요소를 중심으로 작성합니다.
- target_criteria와 target_defect_type에 맞게 충족/불가/검토 상황을 의도적으로 설계합니다.

[충족]
- PEP 계획 항목·수치·구조가 RPT에 빠짐없이, 모순 없이, 측정 가능하게 반영되도록 작성합니다.
- 검증가능성만 다루는 절에서는 RPT 자체가 수치·건수·기준일·점검 결과 등 확인 가능한 근거를 포함하도록 작성합니다.

[완전불가]
- PEP의 목록형 과업·업무범위·대책·절차·산출물 중 정확히 1개만 RPT 어디에도 등장하지 않게 만듭니다.
- 나머지 요구 항목은 정상 반영합니다.
- 단일값 필드의 누락이 아니라 과업, 업무범위, 표준화 항목, 보안대책, 절차, 산출물 등 내용 항목의 누락으로 만듭니다.

[정확불가]
- PEP의 구체 수치·규격·버전·등급·기간을 RPT에서 다르게 기재합니다.
- 또는 RPT 내부에서 실측치 X를 명시한 뒤 목표 Y를 충족했다고 쓰되, X가 Y에 못 미치도록 만듭니다.
- 예: PEP는 TLS 1.3 적용 → RPT는 TLS 1.2 적용, PEP 목표 98% 이상 → RPT 실측 97.8%인데 목표 충족이라고 기재.

[검증불가]
- PEP가 정량 목표나 확인 가능한 기준을 요구했는데 RPT는 "적절히", "충분히", "성실히", "무리 없이", "필요한 범위에서", "우수한 수준으로" 같은 모호 표현으로만 작성합니다.
- 구조나 항목은 존재하되 실측 수치·점검 결과·기준일·건수 등 측정 근거가 없도록 만듭니다.

[추적불가]
- trace_type이 "① 오소·고아 산출물"이면:
  - RPT에 산출물명이 등장하지만 PEP의 어느 계획 단계·과업·Task에도 대응되지 않도록 만듭니다.
  - 계획에 없던 산출물이 갑자기 결과에 나타나거나, 계획 단계와 연결고리 없이 나열되게 작성합니다.

- trace_type이 "② 단계 재구성"이면:
  - PEP의 단계명(A/B/C/D)을 RPT에서 다른 명칭 체계(기능영역 중심, 조직 중심, 화면 단위 중심 등)로 재편합니다.
  - 최소 1개 핵심 단계가 어느 RPT 항목과도 대응되지 않게 만듭니다.

[완전검토]
- PEP 자체를 일부만 제공하거나, RPT가 "부록 참조", "별첨에 수록"이라고만 쓰고 세부 내용을 생략합니다.
- 또는 빠진 항목이 PEP에서 조건부 문구로 되어 있어 필수 여부가 모호하도록 만듭니다.

[정확검토]
- RPT에 수치나 규격은 있으나 PEP, 계약서, 정산자료, 별첨 등이 없어 일치 여부를 확인할 수 없도록 만듭니다.
- 또는 PEP가 "최신 버전", "관련 지침"처럼 비교 기준값을 구체적으로 제시하지 않아 정확성 판단 대상이 애매한 케이스를 만듭니다.

[검증검토]
- 같은 절 안에서 일부 하위항목은 수치·건수·기준일 등 정량 근거를 제시하고, 일부 하위항목은 모호 표현만 사용합니다.
- 또는 RPT가 부록·별첨을 참조하지만 해당 내용이 입력에 없어 일부만 확인 가능하게 만듭니다.

[추적검토]
- 일부 단계·Task·산출물은 PEP와 RPT 간 연결이 확인되지만, 일부는 일정표·산출물표·부록 등이 입력에 없어 전체 추적 가능 여부가 불명확하게 만듭니다.

[COMP/VER 경계]
- PEP에 계획된 구조·절차·항목 자체가 RPT에 없으면 완전성 위반입니다.
- 항목은 있으나 결과·성과·이행 수준이 모호하면 검증가능성 위반입니다.
- 둘 다 해당하도록 설계한 경우 output.코멘트에서 두 기준을 모두 다룹니다.

4단계. output 작성
- 시스템 프롬프트의 [평가기준 정의], [다기준 평가 규칙], [종합판정 규칙], [근거(코멘트) 작성 규칙], [출력 형식]을 그대로 적용합니다.
- output은 반드시 JSON 객체로 작성합니다.
- output.판정 값은 "충족", "불가", "검토" 중 하나만 사용합니다.
- output.판정 값에는 괄호 설명을 붙이지 않습니다.
- 목표 결함 유형의 세분 명칭(완전불가, 정확불가, 검증불가, 추적불가, 완전검토 등)은 생성 설계용 내부 라벨이며 output.판정에는 노출하지 않습니다.
- 완전불가, 정확불가, 검증불가, 추적불가 → output.판정은 "불가"로 씁니다.
- 완전검토, 정확검토, 검증검토, 추적검토 → output.판정은 "검토"로 씁니다.
- target_criteria가 2개 이상이면 결함 없는 기준도 반드시 "문제없음", "누락 없음", "모순 없음", "검증 가능", "판단 대상 아님" 등으로 확인합니다.
- 정확성에서 비교할 기준값 자체가 없는 경우 "판단 대상 아님"으로 구분하고, 억지로 충족/불가로 몰지 않습니다.
- 코멘트는 근거(PEP 인용) → 대조(RPT 인용) → 기준별 판단 → 종합판정 근거 순으로 작성합니다.
- 기준이 1개인 경우에는 자연스러운 한 문단 코멘트를 허용합니다.
- 기준이 2개 이상인 경우에는 기준별 번호 형식을 권장합니다.
- 코멘트에는 "종합적으로 ○○ 기준에서 결함이 확인되므로 최종판정 규칙에 따라 불가로 판정함" 또는 "종합적으로 결함이 확인되지 않아 충족으로 판정함"과 같은 종합 문장을 포함합니다.

5단계. 자체 검증
- output.판정 값이 "충족", "불가", "검토" 중 하나인지 확인합니다.
- output.판정 값에 괄호 설명이 붙어 있지 않은지 확인합니다.
- output.판정 값에 "완전불가", "정확불가", "검증불가", "추적불가", "완전검토", "정확검토", "검증검토", "추적검토" 같은 세분 라벨이 노출되지 않았는지 확인합니다.
- output.코멘트에 등장하는 모든 인용 수치·문구가 PEP_excerpt 또는 RPT_excerpt 원문에 실제로 존재하는지 확인합니다.
- description의 설명 문구, PEP_mapping_note, standard_structure를 실제 원문 근거처럼 인용하지 않았는지 확인합니다.
- input.description이 문자열이 아니라 객체 구조인지 확인합니다.
- input.description이 target_description과 동일한 객체 구조로 출력되었는지 확인합니다.
- input.description.id가 평가 대상 RPT 코드인지 확인합니다.
- input.description.PEP_mapping이 대조 대상 PEP 코드인지 확인합니다.
- target_criteria가 target_description.quality 안에 모두 포함되어 있는지 확인합니다.
- target_description에 없는 평가기준, PEP 코드, RPT 코드를 임의로 추가하지 않았는지 확인합니다.
- 평가기준명이 "완전성", "정확성", "검증가능성", "추적성" 중 하나로만 작성되었는지 확인합니다.
- "검증"이라고 쓴 항목이 남아 있으면 "검증가능성"으로 수정합니다.
- "완전"이라고 쓴 항목이 남아 있으면 "완전성"으로 수정합니다.
- "정확"이라고 쓴 항목이 남아 있으면 "정확성"으로 수정합니다.
- 완전성/검증가능성이 동시에 요청된 경우 구조 부재와 표현 모호를 혼동하지 않았는지 확인합니다.
- 추적성 예시라면 유형(① 오소·고아 산출물 / ② 단계 재구성)과 실제 설계가 일치하는지 확인합니다.
- PEP_mapping이 비어 있는 절에서 완전성·정확성·추적성 시나리오를 만들지 않았는지 확인합니다.
- 목표 결함 유형과 실제 설계한 결함이 논리적으로 일치하는지 확인합니다.
- 불일치가 있으면 해당 예시는 폐기하고 다시 생성합니다.

[출력 스키마]
반드시 아래 형태의 JSON 객체 하나만 반환한다. 다른 텍스트·설명·마크다운을 절대 포함하지 않는다.
{
  "items": [
    {
      "id": "RPT-SEED-XX",
      "input": {
        "description": {
          "id": "RPT-XX-XX",
          "chapter": "제X장 ...",
          "section": "제X절 ...",
          "description": "...",
          "standard_structure": ["..."],
          "quality": ["완전성"],
          "PEP_mapping": ["PEP-XX"],
          "PEP_mapping_note": "..."
        },
        "PEP_excerpt": "...",
        "RPT_excerpt": "..."
      },
      "output": {
        "판정": "충족",
        "코멘트": "..."
      }
    }
  ]
}

[생성 개수]
- 1회 호출당 N개를 생성합니다. 기본값은 5개입니다.
- target_description, target_criteria, target_defect_type이 서로 겹치지 않도록 다양화합니다.
- 기존 분포표를 참고하되, 매트릭스 검증은 프롬프트가 아니라 코드의 RPT_DESCRIPTION_MAP과 target_description으로 처리합니다.
- 충족·불가·검토 라벨 균형을 맞춥니다.
- 추적성 불가 유형은 ① 오소·고아 산출물과 ② 단계 재구성이 한쪽에 치우치지 않도록 균형 있게 생성합니다.
- 추적성 검토 유형 중 "일정/산출물/부록 섹션 자체가 입력에 없는" 케이스도 우선 생성 대상에 포함합니다."""

In [14]:
def build_user_message(desc_code, target_criteria, target_defect_type, trace_type, existing_ids, n=BATCH_N):
    payload = {
        "target_description": RPT_DESCRIPTION_MAP[desc_code],
        "target_criteria": target_criteria,
        "target_defect_type": target_defect_type,
        "existing_ids": sorted(existing_ids),
        "n": n,
    }
    if trace_type:
        payload["trace_type"] = trace_type
    return json.dumps(payload, ensure_ascii=False)


def call_generator(desc_code, target_criteria, target_defect_type, trace_type, existing_ids, n=BATCH_N, retries=3):
    user_msg = build_user_message(desc_code, target_criteria, target_defect_type, trace_type, existing_ids, n)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=0.7,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
            )
            raw = resp.choices[0].message.content
            parsed = json.loads(raw)
            items = parsed.get("items", [])
            if isinstance(items, list) and items:
                return items
        except Exception as e:
            print(f"  [warn] 호출 실패(attempt {attempt+1}): {e}")
            time.sleep(2)
    return []

In [15]:
VALID_VERDICTS = {"충족", "불가", "검토"}
BANNED_SUBLABELS = ["완전불가", "정확불가", "검증불가", "추적불가", "완전검토", "정확검토", "검증검토", "추적검토"]

def validate_item(item, desc_code, target_criteria, existing_ids):
    desc = RPT_DESCRIPTION_MAP[desc_code]

    if not isinstance(item, dict) or not {"id", "input", "output"}.issubset(item.keys()):
        return False, "최상위 필드 누락"
    if item['id'] in existing_ids:
        return False, "id 중복"

    inp = item.get('input', {})
    if not isinstance(inp, dict) or not {"description", "PEP_excerpt", "RPT_excerpt"}.issubset(inp.keys()):
        return False, "input 필드 누락"

    d = inp['description']
    if not isinstance(d, dict):
        return False, "input.description이 객체가 아님"
    if d.get('id') != desc['id']:
        return False, "input.description.id가 target_description과 불일치"
    if set(d.get('PEP_mapping', [])) - set(desc.get('PEP_mapping', [])):
        return False, "target_description.PEP_mapping에 없는 PEP 코드 사용"
    if not set(target_criteria).issubset(set(desc.get('quality', []))):
        return False, "target_criteria가 quality에 포함되지 않음"

    out = item.get('output', {})
    if not isinstance(out, dict) or not {"판정", "코멘트"}.issubset(out.keys()):
        return False, "output 필드 누락"

    verdict = out['판정']
    if verdict not in VALID_VERDICTS:
        return False, f"판정 값 이상함: {verdict}"
    if "(" in verdict or ")" in verdict:
        return False, "판정 값에 괄호 포함"

    for term in BANNED_SUBLABELS:
        if term == verdict:
            return False, f"판정 필드에 세분 라벨 노출: {verdict}"

    return True, "ok"


def numeric_grounding_check(item):
    """코멘트에 등장하는 숫자/버전 패턴이 PEP_excerpt+RPT_excerpt 원문에 있는지 러프하게 확인 (경고용, 자동폐기 아님)"""
    inp = item['input']
    source_text = inp.get('PEP_excerpt', '') + inp.get('RPT_excerpt', '')
    comment = item['output'].get('코멘트', '')
    tokens = re.findall(r'[A-Za-z]+[\s\-]?\d+(?:\.\d+)?%?|\d+(?:\.\d+)?%|\d+(?:일|시간|명|개|분|년|건)', comment)
    missing = [t for t in tokens if t not in source_text]
    return missing

In [16]:
augmented = []
call_count = 0
combo_idx = 0

def current_total():
    return len(seed_data) + len(augmented)

while current_total() < TARGET_TOTAL and call_count < MAX_CALLS:
    combo = combos[combo_idx % len(combos)]
    combo_idx += 1
    desc_code = combo['desc_code']
    target_criteria = combo['target_criteria']
    target_defect_type = combo['target_defect_type']
    trace_type = combo['trace_type']

    remaining = TARGET_TOTAL - current_total()
    n = min(BATCH_N, remaining)
    label = f"{desc_code} / {target_criteria} / {target_defect_type}"
    if trace_type:
        label += f" / {trace_type}"
    print(f"[{call_count+1}] {label} / 요청 {n}개 (누적 {current_total()}/{TARGET_TOTAL})")

    items = call_generator(desc_code, target_criteria, target_defect_type, trace_type, existing_ids, n=n)
    call_count += 1

    accepted = 0
    for item in items:
        ok, reason = validate_item(item, desc_code, target_criteria, existing_ids)
        if not ok:
            print(f"    폐기: {item.get('id','?')} - {reason}")
            continue
        missing = numeric_grounding_check(item)
        if missing:
            print(f"    [주의] {item.get('id','?')} - 코멘트 인용 수치가 원문에서 안 보임(수동확인 필요): {missing}")
        augmented.append(item)
        existing_ids.add(item['id'])
        accepted += 1
    print(f"    -> {accepted}/{len(items)}개 채택")

    with open('augmented_progress.json', 'w', encoding='utf-8') as f:
        json.dump(augmented, f, ensure_ascii=False, indent=2)

    time.sleep(1)

print(f"\n최종 생성: {len(augmented)}개 (전체 {current_total()}/{TARGET_TOTAL})")

[1] RPT-03-01 / ['검증가능성'] / 검증검토 / 요청 5개 (누적 30/150)
    -> 5/5개 채택
[2] RPT-02-04-01 / ['정확성', '검증가능성', '완전성'] / 완전불가 / 요청 5개 (누적 35/150)
    -> 5/5개 채택
[3] RPT-02-02 / ['완전성', '검증가능성', '추적성', '정확성'] / 완전불가 / 요청 5개 (누적 40/150)
    -> 5/5개 채택
[4] RPT-02-01 / ['완전성', '추적성', '검증가능성'] / 추적검토 / 요청 5개 (누적 45/150)
    -> 5/5개 채택
[5] RPT-01-02 / ['완전성', '정확성', '검증가능성'] / 완전검토 / 요청 5개 (누적 50/150)
    -> 5/5개 채택
[6] RPT-02-04-01 / ['정확성', '검증가능성', '완전성'] / 검증불가 / 요청 5개 (누적 55/150)
    -> 5/5개 채택
[7] RPT-02-06 / ['검증가능성'] / 충족 / 요청 5개 (누적 60/150)
    -> 5/5개 채택
[8] RPT-02-02 / ['완전성', '검증가능성', '추적성', '정확성'] / 검증불가 / 요청 5개 (누적 65/150)
    -> 5/5개 채택
[9] RPT-01-04 / ['완전성', '검증가능성'] / 완전불가 / 요청 5개 (누적 70/150)
    -> 5/5개 채택
[10] RPT-02-04-01 / ['정확성', '검증가능성', '완전성'] / 완전검토 / 요청 5개 (누적 75/150)
    -> 5/5개 채택
[11] RPT-01-02 / ['완전성', '정확성', '검증가능성'] / 정확검토 / 요청 5개 (누적 80/150)
    -> 5/5개 채택
[12] RPT-02-02 / ['완전성', '검증가능성', '추적성', '정확성'] / 추적불가 / ① 오소·고아 산출물 / 요청 5개 (누적 85/150)
    -> 5/5개 채택
[13] R

In [17]:
final_dataset = seed_data + augmented
with open('RPT_AUGMENTED_150.json', 'w', encoding='utf-8') as f:
    json.dump(final_dataset, f, ensure_ascii=False, indent=2)

print(f"최종 데이터셋 크기: {len(final_dataset)}개")
from collections import Counter
print("판정 분포:", Counter(d['output']['판정'] for d in final_dataset))
print("description.id 분포:")
for k, v in Counter(d['input']['description']['id'] for d in final_dataset).items():
    print(' ', k, v)

최종 데이터셋 크기: 150개
판정 분포: Counter({'불가': 73, '검토': 46, '충족': 30, '검토(확인필요)': 1})
description.id 분포:
  RPT-01-01 3
  RPT-01-02 19
  RPT-01-03 1
  RPT-01-04 17
  RPT-02-01 24
  RPT-02-02 24
  RPT-02-03-01 8
  RPT-02-04-01 29
  RPT-02-05 9
  RPT-03-01 11
  RPT-02-06 5
